In [15]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from konlpy.tag import Komoran
from collections import Counter
from tqdm import tqdm       #tqdm : 진행 상태를 로그로 표시하는 기능

In [33]:
df = pd.read_csv("./ratings_train.txt", sep='\t')

df.dropna(inplace=True)
df.drop_duplicates('document',inplace=True)

df=df[:5000]
len(df)

5000

In [34]:
df['label'].value_counts()

label
0    2502
1    2498
Name: count, dtype: int64

In [ ]:
komoran=Komoran()
# tokenized_sentence=[komoran.morphs(text) for text in df['document']]
#모든 품사를 사용하니 학습의 능력이 떨어짐
#품사를 필터링
def tokenize(text):
    allow_pos=['NNP','NNG','VV','VA','MAG','SL']
    result=[]
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            result.append(word)
    return result

tokenized_sentence=[tokenize(text) for text in df['document']]

In [37]:
#단어 사전 생성
#패딩 토큰, 언노운 토큰 생성 (초기값)
vocab={
    '<PAD>':0,
    '<UNK>':1
}
#tokenized_sentence에서 모든 토큰을 하나씩 생성
all_tokens=[token for tokens in tokenized_sentence for token in tokens]
#token들의 빈도수를 확인 : min_count로 재한
token_counts=Counter(all_tokens)
token_counts

Counter({'영화': 1770,
         '보': 1334,
         '없': 535,
         '하': 506,
         '좋': 371,
         '있': 337,
         '정말': 323,
         '너무': 311,
         '같': 303,
         '안': 286,
         '진짜': 285,
         '재밌': 282,
         '만들': 264,
         '나오': 239,
         '연기': 237,
         '잘': 210,
         '평점': 205,
         '되': 204,
         '최고': 203,
         '때': 200,
         '왜': 191,
         '사람': 189,
         '다': 183,
         '드라마': 175,
         '스토리': 161,
         '말': 160,
         '이': 159,
         '감동': 157,
         '배우': 157,
         '생각': 156,
         '알': 154,
         '내용': 153,
         '아깝': 149,
         '감독': 143,
         '시간': 142,
         '나': 140,
         '더': 138,
         '이렇': 137,
         '그냥': 135,
         '좀': 134,
         '!!': 132,
         '재미없': 132,
         '재미있': 124,
         '가': 123,
         '재미': 117,
         '모르': 112,
         '남': 106,
         '작품': 106,
         '쓰레기': 105,
         '사랑': 102,
         '쓰':

In [38]:
#단어의 빈도수가 3이상인 토큰들 만을 이용하여 단어 사전에 넣어준다.
for token, count in token_counts.items():
    if count >= 3:
        vocab['token'] = len(vocab)

vocab

{'<PAD>': 0, '<UNK>': 1, 'token': 3}

In [39]:
#vocab을 이용한 토큰화된 데이터의 인코딩과 Dataset을 결합
#dict.get() : 특정 키를 입력하면 해당 키의 값을 되돌려주는 함수 
#               (두번재 인자 값을 이용하여 첫번쨰 인자의 키 값이 존재하지 않을때 디폴트 값을 생성)
vocab.get('마케팅',vocab['<UNK>'])

1

In [40]:
#Dataset을 선언
class RNNDataset(Dataset):
    #생성자, 길이 출력함수, 특정 위치의 데이터 출력함수
    def __init__(self, tokenized_texts, labels, vocab):
        #tokenized_texts : 토큰화된 문서 (독립변수)
        #labels : 정답 데이터 (종속변수)
        #vocab : 단어 사전
        self.labels=labels.values
        self.data=[
            [
                vocab.get(token, vocab['<UNK>']) for token in tokens
            ]
            for tokens in tokenized_texts
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        #getitem 함수의 역할 : DataLoader가 데이터를 불러오는 함수 (독립 변수, 종속 뱐수)
        # return super().__getitem__(idx)
        return torch.tensor(self.data[idx],dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [41]:
#후처리 가공 함수 (DataLoader가 배치 사이즈만큼 Dataset을 불러온 후 데이터를 후처리 가공)
def collate_fn(batch):
    #배치 단위로 들어온 데이터를 최대 길이의 data에 맞게 패딩 토큰을 채워준다.
    #베치 : [ (data, label), (data, label), ... ]
    text_list=[item[0] for item in batch]
    label_list=[item[1] for item in batch]

    #text_list에 있는 인코딩된 데이터에서 최대 길이만큼 나머지 데이터에 패딩 토큰을 채워준다.
    padded_texts=pad_sequence(text_list, batch_first=True, padding_value=vocab['<PAD>'])
    labels=torch.tensor(label_list, dtype=torch.long)

    return padded_texts, labels

In [42]:
#Dataset 셍성
dataset=RNNDataset(tokenized_sentence, df['label'], vocab)
#train의 길이와 test의 길이를 설정
train_size=int(len(dataset) * 0.8)      #int() 사용하는 이유? 길이를 의미하기 때문에 정수형으로 변환 (버림)
test_size=len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, test_size])

print(len(train_dataset), len(val_dataset))

4000 1000


In [43]:
train_loader=DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
val_loader=DataLoader(val_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [44]:
class RNNCLF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes):
        #vocab_size : 임베딩 함수 입력 차원의 수
        #emb_dim : 임베딩 함수 출력 차원의 수
        #hidden_size : RNN 은닉층의 차원의 수
        #num_classes : 선형 모델의 출력 차원의 수 (분류 갯수)
        super().__init__()
        #입력되는 데이터는 인코딩 된 데이터 (2,3,4) -> 벡터화 작업(nn.Embedding(), Word2Vec, FastText, Doc2Vec)
        self.emb=nn.Embedding(vocab_size, emb_dim, padding_idx=vocab['<PAD>'])
        #RNN모델
        self.rnn=nn.RNN(emb_dim, hidden_size, batch_first=True)
        #선형 모델
        self.fc=nn.Linear(hidden_size, num_classes)


    def forward(self,x):
        #x : DataLoader의 독립 변수 값 (토쿤화 데이터)
        embedding=self.emb(x)   #[batch_size, seq_len, emb_dim]

        #rnn_out : [batch_size, seq_len, hidden_size] (모든 시점의 출력)
        #hidden : [1, seq_len, hidden_size] (제일 마지막 시점의 은닉 상태)
        rnn_out, hidden = self.rnn(embedding)

        #선형 모델의 데이터를 대입하기 위해서 hidden의 배치층을 제거
        last_hidden=hidden.squeeze(0)   #[seq_len, hidden]
        
        return self.fc(last_hidden)

In [ ]:
#모덿 생성
model=RNNCLF(len(vocab), emb_dim=64, hidden_size=128, num_classes=2)
#손실 함수
criterion=nn.CrossEntropyLoss()
#옵티마이저
optimizer=optim.Adam(model.parameters(), lr=0.01)

In [46]:
epochs=50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0
    total_train = 0
    #tqdm() : desc는 로그 출력의 값
    for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1} / {epochs} train : '):
        optimizer.zero_grad()
        output=model(inputs)
        loss=criterion(output, labels)       #?
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred=torch.argmax(output, dim=1)
        correct_train += (pred == labels).sum().item()
        total_train+=labels.size(0)

        train_acc=(correct_train / total_train) * 100
        avg_train_loss=train_loss / total_train

        #검증 구간
        model.eval()
        val_loss=0
        correct_val=0
        total_val=0

        with torch.no_grad():
            for inputs,labels in val_loader:
                output=model(inputs)
                loss=criterion(output, labels)

                val_loss+=loss.item()
                pred=torch.argmax(output, dim=1)
                correct_val += (pred  == labels).sum().item()
                total_val+=labels.size(0)
        val_acc=(correct_val / total_val) * 100
        avg_val_loss=val_loss / len(val_loader)
        if (epoch+1)%10 == 0:
            print(f'RNN Epoch : Train Loss : {round(avg_train_loss, 4)} Train Acc : {train_acc}')
            print(f'RNN Epoch : Val Loss : {round(avg_val_loss, 4)} Val Acc : {val_acc}')

Epoch 10 / 50 train :   2%|▏         | 1/63 [00:00<00:16,  3.70it/s]

RNN Epoch : Train Loss : 0.0108 Train Acc : 53.125
RNN Epoch : Val Loss : 0.75 Val Acc : 49.5


Epoch 10 / 50 train :   3%|▎         | 2/63 [00:01<00:58,  1.05it/s]

RNN Epoch : Train Loss : 0.0107 Train Acc : 56.25
RNN Epoch : Val Loss : 0.8378 Val Acc : 49.2


Epoch 10 / 50 train :   5%|▍         | 3/63 [00:02<00:59,  1.01it/s]

...

RNN Epoch : Train Loss : 0.0114 Train Acc : 49.92315573770492
RNN Epoch : Val Loss : 0.7707 Val Acc : 50.1


Epoch 50 / 50 train :  98%|█████████▊| 62/63 [00:19<00:00,  3.07it/s]

RNN Epoch : Train Loss : 0.0114 Train Acc : 49.94959677419355
RNN Epoch : Val Loss : 0.7465 Val Acc : 50.1


Epoch 50 / 50 train : 100%|██████████| 63/63 [00:19<00:00,  3.21it/s]

RNN Epoch : Train Loss : 0.0115 Train Acc : 49.975
RNN Epoch : Val Loss : 0.7008 Val Acc : 50.1
